In [1]:
import pandas as pd
import numpy as np
import requests
import time
from sklearn.neighbors import BallTree

In [2]:
import os
import pandas as pd

files = [
    "Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",
    "Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",
    "Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",
    "Resale flat prices based on registration date from Jan-2017 onwards.csv",
]

folder_path = os.path.join("..", "data")
csv_files = [os.path.join(folder_path, f) for f in files]

df_list = []

for file in csv_files:
    df = pd.read_csv(file)
    df_list.append(df)

df_hdb = pd.concat(df_list, ignore_index=True)
print(df_hdb.shape)


(685478, 11)


In [3]:
df_hdb

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2000-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,NaN
1,2000-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,NaN
2,2000-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,NaN
3,2000-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,NaN
4,2000-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
685473,2026-02,YISHUN,EXECUTIVE,292,YISHUN ST 22,01 TO 03,165.0,Apartment,1992,940000.0,65 years 05 months
685474,2026-02,YISHUN,EXECUTIVE,258,YISHUN ST 22,01 TO 03,148.0,Maisonette,1985,800000.0,58 years 04 months
685475,2026-01,YISHUN,EXECUTIVE,643,YISHUN ST 61,10 TO 12,142.0,Apartment,1987,825000.0,60 years 09 months
685476,2026-01,YISHUN,EXECUTIVE,643,YISHUN ST 61,04 TO 06,146.0,Maisonette,1987,788000.0,60 years 08 months


In [4]:
# Assuming your HDB dataframe is called df_hdb
# 1. Convert Month to Datetime and extract Year, quarter
df_hdb['month'] = pd.to_datetime(df_hdb['month'])
df_hdb['year'] = df_hdb['month'].dt.year
df_hdb['quarter'] = df_hdb['month'].dt.quarter

# 2. Extract numeric Remaining Lease (in years)
def clean_lease(lease_str):
    if isinstance(lease_str, str):
        parts = lease_str.split()
        years = int(parts[0])
        # Add months as a fraction of a year
        months = int(parts[2]) if len(parts) > 2 else 0
        return years + (months / 12)
    return lease_str

df_hdb['remaining_lease_years'] = df_hdb['remaining_lease'].apply(clean_lease)


# 3. Create Address column for Geocoding (Block + Street)
df_hdb['full_address'] = df_hdb['block'] + " " + df_hdb['street_name']

# 4. Log-transform Price (Common in Hedonic Models to normalize data)
df_hdb['log_resale_price'] = np.log(df_hdb['resale_price'])

df_hdb.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,year,quarter,remaining_lease_years,full_address,log_resale_price
0,2000-01-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,NaN,2000,1,NaN,170 ANG MO KIO AVE 4,11.898188
1,2000-01-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,NaN,2000,1,NaN,174 ANG MO KIO AVE 4,11.877569
2,2000-01-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,NaN,2000,1,NaN,216 ANG MO KIO AVE 1,11.976659
3,2000-01-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,NaN,2000,1,NaN,215 ANG MO KIO AVE 1,12.025749
4,2000-01-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,NaN,2000,1,NaN,218 ANG MO KIO AVE 1,12.001505


In [5]:

SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"

def geocode_onemap(search_val, session=None, auth_token=None, retries=3, sleep=0.3):
    s = session or requests.Session()
    headers = {"Authorization": auth_token} if auth_token else {}
    params = {
        "searchVal": search_val,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    for i in range(retries):
        try:
            r = s.get(SEARCH_URL, params=params, headers=headers, timeout=15)
            r.raise_for_status()
            j = r.json()
            if int(j.get("found", 0)) > 0:
                best = j["results"][0]
                return float(best["LATITUDE"]), float(best["LONGITUDE"])
            return None, None
        except Exception:
            if i == retries - 1:
                return None, None
            time.sleep((i + 1) * sleep)



In [6]:
# cache unique addresses
addr_unique = df_hdb["full_address"].dropna().unique()
addr_map = {}
sess = requests.Session()

for a in addr_unique:
    addr_map[a] = geocode_onemap(a, session=sess)

df_hdb["lat"] = df_hdb["full_address"].map(lambda x: addr_map.get(x, (None, None))[0])
df_hdb["lon"] = df_hdb["full_address"].map(lambda x: addr_map.get(x, (None, None))[1])
df_hdb.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,year,quarter,remaining_lease_years,full_address,log_resale_price,lat,lon
0,2000-01-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,NaN,2000,1,NaN,170 ANG MO KIO AVE 4,11.898188,1.374001,103.836432
1,2000-01-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,NaN,2000,1,NaN,174 ANG MO KIO AVE 4,11.877569,1.375097,103.837619
2,2000-01-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,NaN,2000,1,NaN,216 ANG MO KIO AVE 1,11.976659,1.366197,103.841505
3,2000-01-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,NaN,2000,1,NaN,215 ANG MO KIO AVE 1,12.025749,1.366558,103.841624
4,2000-01-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,NaN,2000,1,NaN,218 ANG MO KIO AVE 1,12.001505,1.365119,103.841742


In [7]:
# check for na in any field
print(df_hdb.isna().sum())

month                         0
town                          0
flat_type                     0
block                         0
street_name                   0
storey_range                  0
floor_area_sqm                0
flat_model                    0
lease_commence_date           0
resale_price                  0
remaining_lease          421854
year                          0
quarter                       0
remaining_lease_years    421854
full_address                  0
log_resale_price              0
lat                        3120
lon                        3120
dtype: int64


In [8]:
# Check to see if the % of null values for each of the town is significant
town_null_pct = (
    df_hdb
    .groupby("town")["lat"]
    .apply(lambda x: x.isna().mean() * 100)
)
town_null_pct.sort_values(ascending=False)

town
BUKIT MERAH        2.807376
CLEMENTI           2.581229
QUEENSTOWN         2.461783
CENTRAL AREA       1.774194
JURONG WEST        1.435113
JURONG EAST        0.944475
GEYLANG            0.879144
ANG MO KIO         0.398765
KALLANG/WHAMPOA    0.275551
WOODLANDS          0.182676
TOA PAYOH          0.098624
BEDOK              0.002400
CHOA CHU KANG      0.000000
BISHAN             0.000000
TAMPINES           0.000000
SERANGOON          0.000000
SENGKANG           0.000000
SEMBAWANG          0.000000
PUNGGOL            0.000000
BUKIT TIMAH        0.000000
PASIR RIS          0.000000
MARINE PARADE      0.000000
BUKIT BATOK        0.000000
HOUGANG            0.000000
BUKIT PANJANG      0.000000
YISHUN             0.000000
Name: lat, dtype: float64

In [9]:
# Removing those rows where we can't get lat and lon since they only consists of 3120 rows and it is roughly 2% of each of the affected towns which is a small subset of out data
df_hdb_clean = df_hdb.dropna(subset=['lat','lon'])

In [10]:
#Checking the null values in remaining lease
df_hdb_clean[df_hdb_clean['remaining_lease'].isna()]

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,year,quarter,remaining_lease_years,full_address,log_resale_price,lat,lon
0,2000-01-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,NaN,2000,1,NaN,170 ANG MO KIO AVE 4,11.898188,1.374001,103.836432
1,2000-01-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,NaN,2000,1,NaN,174 ANG MO KIO AVE 4,11.877569,1.375097,103.837619
2,2000-01-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,NaN,2000,1,NaN,216 ANG MO KIO AVE 1,11.976659,1.366197,103.841505
3,2000-01-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,NaN,2000,1,NaN,215 ANG MO KIO AVE 1,12.025749,1.366558,103.841624
4,2000-01-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,NaN,2000,1,NaN,218 ANG MO KIO AVE 1,12.001505,1.365119,103.841742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421849,2014-12-01,YISHUN,5 ROOM,816,YISHUN ST 81,10 TO 12,122.0,Improved,1988,580000.0,NaN,2014,4,NaN,816 YISHUN ST 81,13.270783,1.411771,103.833368
421850,2014-12-01,YISHUN,EXECUTIVE,325,YISHUN CTRL,10 TO 12,146.0,Maisonette,1988,540000.0,NaN,2014,4,NaN,325 YISHUN CTRL,13.199324,1.429239,103.842146
421851,2014-12-01,YISHUN,EXECUTIVE,618,YISHUN RING RD,07 TO 09,164.0,Apartment,1992,738000.0,NaN,2014,4,NaN,618 YISHUN RING RD,13.511699,1.418735,103.835703
421852,2014-12-01,YISHUN,EXECUTIVE,277,YISHUN ST 22,07 TO 09,152.0,Maisonette,1985,592000.0,NaN,2014,4,NaN,277 YISHUN ST 22,13.291262,1.437918,103.836995


In [11]:
# We assume that the lease commence on Jan 1st of each of the years

# convert month column to datetime
df_hdb_clean["month"] = pd.to_datetime(df_hdb_clean["month"], format="%Y-%m-%d")

# assume lease started on Jan 1 of lease_commence_date
lease_start = pd.to_datetime(df_hdb_clean["lease_commence_date"].astype(str) + "-01-01")

# compute remaining lease (years)
remaining_lease_years = 99 - ((df_hdb_clean["month"] - lease_start).dt.days / 365.25)

df_hdb_clean["remaining_lease_years"] = df_hdb_clean["remaining_lease_years"].fillna(remaining_lease_years)

C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\3985060827.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["month"] = pd.to_datetime(df_hdb_clean["month"], format="%Y-%m-%d")
C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\3985060827.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["remaining_lease_years"] = df_hdb_clean["remaining_lease_years"].fillna(remaining_lease_years)


In [12]:
print(df_hdb_clean.isna().sum())

month                         0
town                          0
flat_type                     0
block                         0
street_name                   0
storey_range                  0
floor_area_sqm                0
flat_model                    0
lease_commence_date           0
resale_price                  0
remaining_lease          418734
year                          0
quarter                       0
remaining_lease_years         0
full_address                  0
log_resale_price              0
lat                           0
lon                           0
dtype: int64


In [13]:
# Remove unnecessary column
df_hdb_clean= df_hdb_clean.drop(columns=['remaining_lease'])

In [14]:
df_hdb_clean

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,quarter,remaining_lease_years,full_address,log_resale_price,lat,lon
0,2000-01-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,2000,1,85.001369,170 ANG MO KIO AVE 4,11.898188,1.374001,103.836432
1,2000-01-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,2000,1,85.001369,174 ANG MO KIO AVE 4,11.877569,1.375097,103.837619
2,2000-01-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,2000,1,75.000000,216 ANG MO KIO AVE 1,11.976659,1.366197,103.841505
3,2000-01-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,2000,1,75.000000,215 ANG MO KIO AVE 1,12.025749,1.366558,103.841624
4,2000-01-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,2000,1,75.000000,218 ANG MO KIO AVE 1,12.001505,1.365119,103.841742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685473,2026-02-01,YISHUN,EXECUTIVE,292,YISHUN ST 22,01 TO 03,165.0,Apartment,1992,940000.0,2026,1,65.416667,292 YISHUN ST 22,13.753635,1.436725,103.836966
685474,2026-02-01,YISHUN,EXECUTIVE,258,YISHUN ST 22,01 TO 03,148.0,Maisonette,1985,800000.0,2026,1,58.333333,258 YISHUN ST 22,13.592367,1.435156,103.839804
685475,2026-01-01,YISHUN,EXECUTIVE,643,YISHUN ST 61,10 TO 12,142.0,Apartment,1987,825000.0,2026,1,60.750000,643 YISHUN ST 61,13.623139,1.421335,103.837437
685476,2026-01-01,YISHUN,EXECUTIVE,643,YISHUN ST 61,04 TO 06,146.0,Maisonette,1987,788000.0,2026,1,60.666667,643 YISHUN ST 61,13.577253,1.421335,103.837437


In [15]:
# Only keeping data from 2009 and onwards since we only have data from 2009 onwards
df_hdb_clean= df_hdb_clean[df_hdb_clean['month']>='2009-01-01']

In [16]:
df_hdb_clean.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,quarter,remaining_lease_years,full_address,log_resale_price,lat,lon
278846,2009-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,188000.0,2009,1,67.999316,314 ANG MO KIO AVE 3,12.144197,1.366227,103.850086
278847,2009-01-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,01 TO 03,60.0,Improved,1986,182000.0,2009,1,75.999316,174 ANG MO KIO AVE 4,12.111762,1.375097,103.837619
278848,2009-01-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,04 TO 06,73.0,New Generation,1976,278000.0,2009,1,65.997947,215 ANG MO KIO AVE 1,12.535376,1.366558,103.841624
278849,2009-01-01,ANG MO KIO,3 ROOM,219,ANG MO KIO AVE 1,10 TO 12,82.0,New Generation,1977,298000.0,2009,1,67.000000,219 ANG MO KIO AVE 1,12.604849,1.365982,103.840654
278850,2009-01-01,ANG MO KIO,3 ROOM,219,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1977,238000.0,2009,1,67.000000,219 ANG MO KIO AVE 1,12.380026,1.365982,103.840654


# Feature engineering

In [17]:
df_hdb_clean["floor_area_sqft"] = df_hdb_clean["floor_area_sqm"] * 10.7639
df_hdb_clean["price_per_sqft"] = df_hdb_clean["resale_price"] / df_hdb_clean["floor_area_sqft"]
df_hdb_clean["log_price"] = np.log(df_hdb_clean["resale_price"])
df_hdb_clean["log_price_per_sqft"] = np.log(df_hdb_clean["price_per_sqft"])

# mature estates indicator
mature_estates = [
    "ANG MO KIO","BEDOK","BISHAN","BUKIT MERAH","BUKIT TIMAH",
    "CENTRAL AREA","GEYLANG","KALLANG/WHAMPOA","MARINE PARADE",
    "QUEENSTOWN","SERANGOON","TOA PAYOH"
]

df_hdb_clean["mature_estate"] = df_hdb_clean["town"].isin(mature_estates).astype(int)


#flat type mapping
flat_type_map = {
    "1 ROOM":1,
    "2 ROOM":2,
    "3 ROOM":3,
    "4 ROOM":4,
    "5 ROOM":5,
    "EXECUTIVE":6,
    "MULTI-GENERATION":7
}

df_hdb_clean["flat_type"] = df_hdb_clean["flat_type"].map(flat_type_map)


# categorise storey range
storey_split = df_hdb_clean["storey_range"].str.split(" TO ", expand=True)

df_hdb_clean["storey_low"] = storey_split[0].astype(int)
df_hdb_clean["storey_high"] = storey_split[1].astype(int)

df_hdb_clean["storey_mid"] = (df_hdb_clean["storey_low"] + df_hdb_clean["storey_high"]) / 2

df_hdb_clean["storey_percentile"] = (
    df_hdb_clean.groupby("town")["storey_mid"]
    .rank(pct=True)
)

df_hdb_clean["storey_relative_category"] = pd.cut(
    df_hdb_clean["storey_percentile"],
    bins=[0,0.33,0.66,1],
    labels=["LOW_IN_ESTATE","MID_IN_ESTATE","HIGH_IN_ESTATE"]
)

cols_to_drop = [
    "storey_low",
    "storey_high",
    "storey_range",
    "storey_mid",
    "storey_percentile"
]

df_hdb_clean.drop(columns=cols_to_drop, errors="ignore", inplace=True)

print(df_hdb_clean.columns)
print(df_hdb_clean.head())

C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\652024431.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["floor_area_sqft"] = df_hdb_clean["floor_area_sqm"] * 10.7639
C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\652024431.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["price_per_sqft"] = df_hdb_clean["resale_price"] / df_hdb_clean["floor_area_sqft"]
C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\652024431.py:3: SettingWithCopyWarning: 
A value is trying to be se

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'floor_area_sqm',
       'flat_model', 'lease_commence_date', 'resale_price', 'year', 'quarter',
       'remaining_lease_years', 'full_address', 'log_resale_price', 'lat',
       'lon', 'floor_area_sqft', 'price_per_sqft', 'log_price',
       'log_price_per_sqft', 'mature_estate', 'storey_relative_category'],
      dtype='object')
            month        town  flat_type block       street_name  \
278846 2009-01-01  ANG MO KIO          2   314  ANG MO KIO AVE 3   
278847 2009-01-01  ANG MO KIO          3   174  ANG MO KIO AVE 4   
278848 2009-01-01  ANG MO KIO          3   215  ANG MO KIO AVE 1   
278849 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1   
278850 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1   

        floor_area_sqm      flat_model  lease_commence_date  resale_price  \
278846            44.0        Improved                 1978      188000.0   
278847            60.0        Improved     

C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\652024431.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["storey_percentile"] = (
C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\652024431.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hdb_clean["storey_relative_category"] = pd.cut(
C:\Users\marcu\AppData\Local\Temp\ipykernel_18748\652024431.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: 

In [18]:
print(df_hdb_clean.head())
print(df_hdb_clean.columns)

            month        town  flat_type block       street_name  \
278846 2009-01-01  ANG MO KIO          2   314  ANG MO KIO AVE 3   
278847 2009-01-01  ANG MO KIO          3   174  ANG MO KIO AVE 4   
278848 2009-01-01  ANG MO KIO          3   215  ANG MO KIO AVE 1   
278849 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1   
278850 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1   

        floor_area_sqm      flat_model  lease_commence_date  resale_price  \
278846            44.0        Improved                 1978      188000.0   
278847            60.0        Improved                 1986      182000.0   
278848            73.0  New Generation                 1976      278000.0   
278849            82.0  New Generation                 1977      298000.0   
278850            67.0  New Generation                 1977      238000.0   

        year  ...          full_address  log_resale_price       lat  \
278846  2009  ...  314 ANG MO KIO AVE 3         12.144197

# adding external factors

In [19]:
hawker_centres = pd.read_csv("../data/cleaned/hawker_centres_cleaned.csv")
bus_stops = pd.read_csv("../data/cleaned/bus_stops_cleaned.csv")
mall = pd.read_csv("../data/cleaned/mall_cleaned.csv")
mrt = pd.read_csv("../data/cleaned/mrt_cleaned.csv")
rpi = pd.read_csv("../data/cleaned/rpi_cleaned.csv")

In [20]:
df_hdb_clean = df_hdb_clean.merge(rpi, on=["quarter", "year"], how="left")
print(df_hdb_clean.head())
print(df_hdb_clean.columns)

       month        town  flat_type block       street_name  floor_area_sqm  \
0 2009-01-01  ANG MO KIO          2   314  ANG MO KIO AVE 3            44.0   
1 2009-01-01  ANG MO KIO          3   174  ANG MO KIO AVE 4            60.0   
2 2009-01-01  ANG MO KIO          3   215  ANG MO KIO AVE 1            73.0   
3 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1            82.0   
4 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1            67.0   

       flat_model  lease_commence_date  resale_price  year  ...  \
0        Improved                 1978      188000.0  2009  ...   
1        Improved                 1986      182000.0  2009  ...   
2  New Generation                 1976      278000.0  2009  ...   
3  New Generation                 1977      298000.0  2009  ...   
4  New Generation                 1977      238000.0  2009  ...   

   log_resale_price       lat         lon  floor_area_sqft  price_per_sqft  \
0         12.144197  1.366227  103.850086   

In [21]:
# calculate real prices


df_hdb_clean["real_price"] = df_hdb_clean["resale_price"] / (df_hdb_clean["Index"] / 100.0)
df_hdb_clean["real_price_psf"] = df_hdb_clean["real_price"] / df_hdb_clean["floor_area_sqft"]

# log prices
df_hdb_clean["log_real_price"] = np.log(df_hdb_clean["real_price"])
df_hdb_clean["log_real_price_psf"] = np.log(df_hdb_clean["real_price_psf"])
df_hdb_clean.head()

,month,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,...,price_per_sqft,log_price,log_price_per_sqft,mature_estate,storey_relative_category,Index,real_price,real_price_psf,log_real_price,log_real_price_psf
0,2009-01-01,ANG MO KIO,2,314,ANG MO KIO AVE 3,44.0,Improved,1978,188000.0,2009,...,396.949737,12.144197,5.983810,1,HIGH_IN_ESTATE,201.0,93532.338308,197.487431,11.446063,5.285675
1,2009-01-01,ANG MO KIO,3,174,ANG MO KIO AVE 4,60.0,Improved,1986,182000.0,2009,...,281.806161,12.111762,5.641219,1,LOW_IN_ESTATE,201.0,90547.263682,140.202070,11.413627,4.943085
2,2009-01-01,ANG MO KIO,3,215,ANG MO KIO AVE 1,73.0,New Generation,1976,278000.0,2009,...,353.795481,12.535376,5.868719,1,LOW_IN_ESTATE,201.0,138308.457711,176.017652,11.837242,5.170584
3,2009-01-01,ANG MO KIO,3,219,ANG MO KIO AVE 1,82.0,New Generation,1977,298000.0,2009,...,337.623570,12.604849,5.821932,1,HIGH_IN_ESTATE,201.0,148258.706468,167.971925,11.906714,5.123797
4,2009-01-01,ANG MO KIO,3,219,ANG MO KIO AVE 1,67.0,New Generation,1977,238000.0,2009,...,330.014103,12.380026,5.799135,1,MID_IN_ESTATE,201.0,118407.960199,164.186121,11.681891,5.101001


In [22]:
# rpi dont have 2026 data, remove 2026 data
df_hdb_clean = df_hdb_clean[df_hdb_clean["year"] < 2026]

In [23]:
EARTH_RADIUS_KM = 6371.0

def add_nearest_distance_km(
    left_df, left_lat, left_lon,
    right_df, right_lat, right_lon,
    out_col
):
    # coordinates in radians: [lat, lon]
    left_coords = np.deg2rad(left_df[[left_lat, left_lon]].to_numpy())
    right_coords = np.deg2rad(right_df[[right_lat, right_lon]].to_numpy())

    tree = BallTree(right_coords, metric="haversine")
    dist_rad, _ = tree.query(left_coords, k=1)
    left_df[out_col] = dist_rad[:, 0] * EARTH_RADIUS_KM
    return left_df


In [24]:
df = df_hdb_clean.copy()

df = add_nearest_distance_km(df, "lat", "lon", hawker_centres, "lat", "lon", "dist_to_nearest_hawker_km")
df = add_nearest_distance_km(df, "lat", "lon", bus_stops,      "lat", "lon", "dist_to_nearest_busstop_km")
df = add_nearest_distance_km(df, "lat", "lon", mall,           "lat", "lon", "dist_to_nearest_mall_km")
df = add_nearest_distance_km(df, "lat", "lon", mrt,            "lat", "lon", "dist_to_nearest_mrt_km")
df.head()

,month,town,flat_type,block,street_name,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,...,storey_relative_category,Index,real_price,real_price_psf,log_real_price,log_real_price_psf,dist_to_nearest_hawker_km,dist_to_nearest_busstop_km,dist_to_nearest_mall_km,dist_to_nearest_mrt_km
0,2009-01-01,ANG MO KIO,2,314,ANG MO KIO AVE 3,44.0,Improved,1978,188000.0,2009,...,HIGH_IN_ESTATE,201.0,93532.338308,197.487431,11.446063,5.285675,0.312359,0.120232,0.406862,0.416219
1,2009-01-01,ANG MO KIO,3,174,ANG MO KIO AVE 4,60.0,Improved,1986,182000.0,2009,...,LOW_IN_ESTATE,201.0,90547.263682,140.202070,11.413627,4.943085,0.184321,0.069824,0.991619,0.420669
2,2009-01-01,ANG MO KIO,3,215,ANG MO KIO AVE 1,73.0,New Generation,1976,278000.0,2009,...,LOW_IN_ESTATE,201.0,138308.457711,176.017652,11.837242,5.170584,0.191689,0.218725,0.749536,0.783192
3,2009-01-01,ANG MO KIO,3,219,ANG MO KIO AVE 1,82.0,New Generation,1977,298000.0,2009,...,HIGH_IN_ESTATE,201.0,148258.706468,167.971925,11.906714,5.123797,0.152224,0.121225,0.869312,0.760135
4,2009-01-01,ANG MO KIO,3,219,ANG MO KIO AVE 1,67.0,New Generation,1977,238000.0,2009,...,MID_IN_ESTATE,201.0,118407.960199,164.186121,11.681891,5.101001,0.152224,0.121225,0.869312,0.760135


In [25]:
# check for na in all col
print(df.isna().sum())


month                         0
town                          0
flat_type                     0
block                         0
street_name                   0
floor_area_sqm                0
flat_model                    0
lease_commence_date           0
resale_price                  0
year                          0
quarter                       0
remaining_lease_years         0
full_address                  0
log_resale_price              0
lat                           0
lon                           0
floor_area_sqft               0
price_per_sqft                0
log_price                     0
log_price_per_sqft            0
mature_estate                 0
storey_relative_category      0
Index                         0
real_price                    0
real_price_psf                0
log_real_price                0
log_real_price_psf            0
dist_to_nearest_hawker_km     0
dist_to_nearest_busstop_km    0
dist_to_nearest_mall_km       0
dist_to_nearest_mrt_km        0
dtype: i

In [26]:
df_hdb_final = df
print(df_hdb_final.head())
print(df_hdb_final.columns)


       month        town  flat_type block       street_name  floor_area_sqm  \
0 2009-01-01  ANG MO KIO          2   314  ANG MO KIO AVE 3            44.0   
1 2009-01-01  ANG MO KIO          3   174  ANG MO KIO AVE 4            60.0   
2 2009-01-01  ANG MO KIO          3   215  ANG MO KIO AVE 1            73.0   
3 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1            82.0   
4 2009-01-01  ANG MO KIO          3   219  ANG MO KIO AVE 1            67.0   

       flat_model  lease_commence_date  resale_price  year  ...  \
0        Improved                 1978      188000.0  2009  ...   
1        Improved                 1986      182000.0  2009  ...   
2  New Generation                 1976      278000.0  2009  ...   
3  New Generation                 1977      298000.0  2009  ...   
4  New Generation                 1977      238000.0  2009  ...   

   storey_relative_category  Index     real_price  real_price_psf  \
0            HIGH_IN_ESTATE  201.0   93532.338308    

In [27]:
df_hdb_final.to_csv("../data/hdb_final.csv", index=False)